# 01 — Explore: Ground Truth + HSV Variance

Run from the `tests/object_detection/` directory:
```
cd tests/object_detection
jupyter notebook 01_explore.ipynb
```

In [ ]:
import sys
import pathlib
import json

import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

HERE = pathlib.Path().resolve()
sys.path.insert(0, str(HERE.parent.parent / 'src'))

frame_paths = sorted(HERE.glob('frame_*.jpg'))
frames_bgr = [cv.imread(str(p)) for p in frame_paths]
print(f'Loaded {len(frames_bgr)} frames from {HERE}')
print(f'Frame size: {frames_bgr[0].shape[1]}x{frames_bgr[0].shape[0]}')

## Visual preview of frames

Quick grid to see the strobe variation.

In [ ]:
cols = 4
rows = (len(frames_bgr) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 3))
for idx, (ax, frame) in enumerate(zip(axes.flat, frames_bgr)):
    ax.imshow(cv.cvtColor(frame, cv.COLOR_BGR2RGB))
    ax.set_title(frame_paths[idx].name, fontsize=8)
    ax.axis('off')
for ax in axes.flat[len(frames_bgr):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(str(HERE / 'preview_grid.jpg'), dpi=80)
plt.show()

## Vision annotation via Claude API

For each frame, send it to `claude-sonnet-4-6` and ask for JSON bounding boxes of every colored box visible. Aggregated into `ground_truth.json`.

In [ ]:
import base64

gt_path = HERE / 'ground_truth.json'
if gt_path.exists():
    all_annotations = json.loads(gt_path.read_text())
    print(f'Loaded existing ground_truth.json ({len(all_annotations)} frames)')
    for i, boxes in enumerate(all_annotations):
        colors = [b['color'] for b in boxes]
        print(f'  frame_{i:02d}: {len(boxes)} boxes — {colors}')
else:
    import anthropic
    client = anthropic.Anthropic()

    SYSTEM = (
        'You are a vision assistant analyzing images from a robotics camera. '
        'The image shows a flat playfield (dark surface) with small colored cubes/boxes on it. '
        'Return ONLY valid JSON — a list of objects, one per colored box you can see. '
        'Each object: {"color": "<name>", "cx": <int>, "cy": <int>, "w": <int>, "h": <int>} '
        'where cx,cy is the pixel center and w,h is the approximate bounding box in pixels. '
        'Use color names from this list: red, orange, yellow, green, teal, cyan, blue, purple. '
        'Output ONLY the JSON array, no markdown, no explanation.'
    )

    def annotate_frame(frame_bgr):
        _, buf = cv.imencode('.jpg', frame_bgr, [cv.IMWRITE_JPEG_QUALITY, 85])
        b64 = base64.standard_b64encode(buf.tobytes()).decode()
        msg = client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=512,
            system=SYSTEM,
            messages=[{
                'role': 'user',
                'content': [{
                    'type': 'image',
                    'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': b64}
                }, {
                    'type': 'text',
                    'text': 'List all colored boxes you see on the playfield.'
                }]
            }]
        )
        text = msg.content[0].text.strip()
        if text.startswith('```'):
            lines = text.split('\n')
            text = '\n'.join(l for l in lines if not l.startswith('```'))
        return json.loads(text)

    all_annotations = []
    for i, frame in enumerate(frames_bgr):
        boxes = annotate_frame(frame)
        all_annotations.append(boxes)
        colors = [b['color'] for b in boxes]
        print(f'frame_{i:02d}: {len(boxes)} boxes — {colors}')

    gt_path.write_text(json.dumps(all_annotations, indent=2))
    print(f'\nSaved to {gt_path}')


## Consensus ground truth

Average box centers across frames to get stable reference positions.

In [ ]:
from collections import defaultdict

color_positions = defaultdict(list)
for frame_boxes in all_annotations:
    for b in frame_boxes:
        color_positions[b['color']].append((b['cx'], b['cy']))

consensus = {}
for color, pts in color_positions.items():
    arr = np.array(pts)
    consensus[color] = {'cx': float(arr[:,0].mean()), 'cy': float(arr[:,1].mean()),
                        'std_cx': float(arr[:,0].std()), 'std_cy': float(arr[:,1].std()),
                        'n': len(pts)}
    print(f'{color:8s}  center=({consensus[color]["cx"]:.0f},{consensus[color]["cy"]:.0f})'
          f'  std=({consensus[color]["std_cx"]:.1f},{consensus[color]["std_cy"]:.1f})'
          f'  seen in {len(pts)}/{len(frames_bgr)} frames')

(HERE / 'consensus.json').write_text(json.dumps(consensus, indent=2))

## Visualize annotations on frame 0

In [ ]:
vis = frames_bgr[0].copy()
COLOR_BGR = {
    'red': (0,0,220), 'orange': (0,128,255), 'yellow': (0,220,220),
    'green': (0,200,0), 'teal': (180,180,0), 'cyan': (220,220,0),
    'blue': (220,0,0), 'purple': (180,0,180),
}
for b in all_annotations[0]:
    cx, cy = int(b['cx']), int(b['cy'])
    hw, hh = b.get('w', 40)//2, b.get('h', 40)//2
    bgr = COLOR_BGR.get(b['color'], (200,200,200))
    cv.rectangle(vis, (cx-hw, cy-hh), (cx+hw, cy+hh), bgr, 2)
    cv.putText(vis, b['color'], (cx-hw, cy-hh-4), cv.FONT_HERSHEY_SIMPLEX, 0.5, bgr, 1)

plt.figure(figsize=(12, 8))
plt.imshow(cv.cvtColor(vis, cv.COLOR_BGR2RGB))
plt.title('Claude vision annotations — frame_00')
plt.axis('off')
plt.savefig(str(HERE / 'annotated_frame00.jpg'), dpi=80)
plt.show()

## HSV variance across strobe cycle

For each consensus box location, sample a 20×20 ROI in every frame. Plot H, S, V distributions to see how much the strobe affects each channel.

In [ ]:
ROI_HALF = 10  # 20x20 px ROI

hsv_by_color = {}  # color -> {'h': [...], 's': [...], 'v': [...]}

for color, info in consensus.items():
    cx, cy = int(info['cx']), int(info['cy'])
    h_vals, s_vals, v_vals = [], [], []
    for frame in frames_bgr:
        y1, y2 = max(0, cy-ROI_HALF), min(frame.shape[0], cy+ROI_HALF)
        x1, x2 = max(0, cx-ROI_HALF), min(frame.shape[1], cx+ROI_HALF)
        roi = frame[y1:y2, x1:x2]
        hsv_roi = cv.cvtColor(roi, cv.COLOR_BGR2HSV).reshape(-1, 3).astype(float)
        h_vals.extend(hsv_roi[:, 0].tolist())
        s_vals.extend(hsv_roi[:, 1].tolist())
        v_vals.extend(hsv_roi[:, 2].tolist())
    hsv_by_color[color] = {'h': h_vals, 's': s_vals, 'v': v_vals}

# Summary table
print(f'{'Color':8s}  H_med±std      S_med±std      V_med±std')
print('-' * 55)
for color, vals in hsv_by_color.items():
    h, s, v = np.array(vals['h']), np.array(vals['s']), np.array(vals['v'])
    print(f'{color:8s}  {np.median(h):5.1f}±{h.std():4.1f}   '
          f'{np.median(s):5.1f}±{s.std():4.1f}   '
          f'{np.median(v):5.1f}±{v.std():4.1f}')

## V-channel variance: box vs background

This is the key chart: how much does V fluctuate across frames at box locations vs. background?

In [ ]:
# Sample per-frame median V at each box location
per_frame_v = {color: [] for color in consensus}
per_frame_v['background'] = []

# Background sample points (center of frame, away from boxes)
bg_pts = [(300, 400), (700, 200), (900, 600)]

for frame in frames_bgr:
    for color, info in consensus.items():
        cx, cy = int(info['cx']), int(info['cy'])
        roi = frame[max(0,cy-ROI_HALF):cy+ROI_HALF, max(0,cx-ROI_HALF):cx+ROI_HALF]
        v = cv.cvtColor(roi, cv.COLOR_BGR2HSV)[:,:,2]
        per_frame_v[color].append(float(np.median(v)))
    # Background
    bg_v = []
    for bx, by in bg_pts:
        roi = frame[by-10:by+10, bx-10:bx+10]
        v = cv.cvtColor(roi, cv.COLOR_BGR2HSV)[:,:,2]
        bg_v.append(float(np.median(v)))
    per_frame_v['background'].append(float(np.mean(bg_v)))

fig, ax = plt.subplots(figsize=(12, 5))
for label, vals in per_frame_v.items():
    ax.plot(vals, marker='o', label=label)
ax.set_xlabel('Frame index')
ax.set_ylabel('Median V value')
ax.set_title('Per-frame V channel at box/background locations (strobe effect)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(HERE / 'v_variance.jpg'), dpi=80)
plt.show()

print('\nV std across frames:')
for label, vals in per_frame_v.items():
    print(f'  {label:12s}: std={np.std(vals):.2f}  range=[{min(vals):.0f},{max(vals):.0f}]')

## Wood grain vs box: HSV distribution comparison

Do S and H separate boxes from wood grain better than V?

In [ ]:
# Sample multiple background patches across the playfield
bg_sample_pts = [
    (200, 200), (400, 300), (600, 400), (800, 500), (1000, 300),
    (300, 600), (700, 650), (500, 150), (900, 400), (150, 500)
]

bg_hsv = {'h': [], 's': [], 'v': []}
for frame in frames_bgr:
    for bx, by in bg_sample_pts:
        roi = frame[max(0,by-10):by+10, max(0,bx-10):bx+10]
        if roi.size == 0:
            continue
        hsv_roi = cv.cvtColor(roi, cv.COLOR_BGR2HSV).reshape(-1, 3).astype(float)
        bg_hsv['h'].extend(hsv_roi[:,0].tolist())
        bg_hsv['s'].extend(hsv_roi[:,1].tolist())
        bg_hsv['v'].extend(hsv_roi[:,2].tolist())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
channel_names = ['H (Hue)', 'S (Saturation)', 'V (Value/Brightness)']
channel_keys = ['h', 's', 'v']
channel_ranges = [(0, 180), (0, 255), (0, 255)]

for ax, ch, cname, crange in zip(axes, channel_keys, channel_names, channel_ranges):
    bg_arr = np.array(bg_hsv[ch])
    bins = np.linspace(crange[0], crange[1], 50)
    ax.hist(bg_arr, bins=bins, alpha=0.5, label='background/wood', color='saddlebrown', density=True)
    for color, vals in hsv_by_color.items():
        arr = np.array(vals[ch])
        ax.hist(arr, bins=bins, alpha=0.5, label=color, density=True)
    ax.set_title(cname)
    ax.set_xlabel('Value')
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

plt.suptitle('HSV distributions: colored boxes vs background/wood grain')
plt.tight_layout()
plt.savefig(str(HERE / 'hsv_distributions.jpg'), dpi=80)
plt.show()

## Summary table

Median HSV ± std for every box color across all 12 frames.

In [ ]:
import pandas as pd

rows = []
for color, vals in hsv_by_color.items():
    h, s, v = np.array(vals['h']), np.array(vals['s']), np.array(vals['v'])
    rows.append({
        'color': color,
        'H_med': round(float(np.median(h)), 1),
        'H_std': round(float(h.std()), 1),
        'S_med': round(float(np.median(s)), 1),
        'S_std': round(float(s.std()), 1),
        'V_med': round(float(np.median(v)), 1),
        'V_std': round(float(v.std()), 1),
    })

df = pd.DataFrame(rows).set_index('color')
print(df.to_string())
df.to_csv(str(HERE / 'hsv_summary.csv'))
print('\nSaved hsv_summary.csv')